In [2]:
!pip install deep_sort_realtime
!pip install ultralytics

In [3]:
from ultralytics import YOLO
import cv2
import numpy as np
import os
import csv
from collections import deque
from deep_sort_realtime.deepsort_tracker import DeepSort

tracker = DeepSort(max_age=10)

/opt/conda/lib/python3.13/site-packages/deep_sort_realtime/embedder/embedder_pytorch.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
model = YOLO("construction_safety_v8.pt")
#model = YOLO("yolov8n.pt") 

In [5]:
def process_video(input_video):
    # 1. STANDARDIZATION SETUP
    TARGET_WIDTH = 1280
    TARGET_HEIGHT = 720
    
    base_name = os.path.splitext(os.path.basename(input_video))[0]
    output_video = f"runs/detect/predict/{base_name}_Annotated.mp4"
    csv_file = f"runs/detect/reports/{base_name}_report.csv"
    csv_idle_file = f"runs/detect/reports/{base_name}_idle_report.csv"
    
    os.makedirs(os.path.dirname(output_video), exist_ok=True)
    os.makedirs(os.path.dirname(csv_file), exist_ok=True)

    # ---------------- Video setup ----------------
    cap = cv2.VideoCapture(input_video)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    # INITIALIZE ONCE OUTSIDE THE LOOP
    out = cv2.VideoWriter(output_video, fourcc, fps, (TARGET_WIDTH, TARGET_HEIGHT))

    # ---------------- CSV setup ----------------
    csv_writer_obj = open(csv_file, mode='w', newline='', buffering=1)
    writer = csv.writer(csv_writer_obj)
    # ADDED "Idle Clusters" to the header
    writer.writerow(["Frame", "Total", "Compliant", "No Vest", "Compliance %", "Idle Clusters"])

    csv_idle_obj = open(csv_idle_file, mode='w', newline='', buffering=1)
    idle_writer = csv.writer(csv_idle_obj)
    idle_writer.writerow(["Frame", "Track ID", "Idle Frames"])

    # ---------------- Load model & tracker ----------------
    model = YOLO("construction_safety_v8.pt")
    tracker = DeepSort(max_age=30)
    track_data = {}

    # ---------------- Parameters ----------------
    MAX_HISTORY = 30
    IDLE_THRESHOLD = 2
    IDLE_FRAME_LIMIT = 30
    ALERT_THRESHOLD = 10
    COMPLIANCE_LIMIT = 70
    PROXIMITY_RATIO = 0.2

    alert_counter = 0
    frame_idx = 0

    # ---------------- Helpers ----------------
    def get_center(x1, y1, x2, y2):
        return ((x1 + x2) // 2, (y1 + y2) // 2)

    def distance(p1, p2):
        return ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)**0.5

    def get_movement(track_info):
        centers = list(track_info.get('history_centers', []))
        # FIX: If we only have 1 point, movement is 0 (stationary), not Infinity or Error
        if len(centers) < 2: 
            return 0.0 
        
        # Calculate average distance between consecutive points
        total_dist = sum(distance(centers[i-1], centers[i]) for i in range(1, len(centers)))
        return total_dist / (len(centers) - 1)
    def get_vest_color(vest_crop):
        if vest_crop.size == 0: return "Unknown"
        vest_crop = cv2.convertScaleAbs(vest_crop, alpha=1.2, beta=10)
        hsv = cv2.cvtColor(vest_crop, cv2.COLOR_BGR2HSV)
        
        y_mask = cv2.inRange(hsv, np.array([20,100,100]), np.array([45,255,255]))
        o_mask = cv2.inRange(hsv, np.array([0,100,100]), np.array([19,255,255]))
        b_mask = cv2.inRange(hsv, np.array([80,40,40]), np.array([140,255,255]))

        y_sum, o_sum, b_sum = np.sum(y_mask>0), np.sum(o_mask>0), np.sum(b_mask>0)
        min_pixels = int(vest_crop.shape[0] * vest_crop.shape[1] * 0.05)

        if y_sum > max(o_sum, b_sum) and y_sum > min_pixels: return "Yellow"
        if o_sum > max(y_sum, b_sum) and o_sum > min_pixels: return "Orange"
        if b_sum > max(y_sum, o_sum) and b_sum > min_pixels: return "Blue"
        return "NO VEST" if (y_sum+o_sum+b_sum) < min_pixels else "Unknown"

    print(f"Processing {input_video} at {TARGET_WIDTH}x{TARGET_HEIGHT}...")

    # ---------------- Main loop ----------------
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break

            # RESIZE FRAME IMMEDIATELY
            frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT))

            frame_total, frame_vests, frame_no_vests = 0, 0, 0
            cluster_count = 0 # Initialize for every frame
            results = model(frame, verbose=False, imgsz=1280)[0]

            detections = []
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls_id = int(box.cls[0])
                label = model.names[cls_id]
                conf = float(box.conf[0])
                if conf > 0.4 and label in ['Safety Vests', 'NO-Safety Vest', 'Safety Vest']:
                    detections.append(([x1, y1, x2-x1, y2-y1], conf, label))

            # --- TRACKING ---
            tracks_out = tracker.update_tracks(detections, frame=frame)

            for track in tracks_out:
                track_id = track.track_id
                
                if track_id not in track_data:
                    track_data[track_id] = {
                        'history': deque(maxlen=MAX_HISTORY), 
                        'history_centers': deque(maxlen=MAX_HISTORY), 
                        'idle_frames': 0
                    }

                ltrb_raw = track.to_tlbr()
                x1, y1, x2, y2 = map(int, ltrb_raw) # Standard coords for logic
                center = get_center(x1, y1, x2, y2)
                track_data[track_id]['history_centers'].append(center)

                # Calculate Movement
                movement = get_movement(track_data[track_id])
                if movement < IDLE_THRESHOLD:
                    track_data[track_id]['idle_frames'] += 1
                else:
                    track_data[track_id]['idle_frames'] = 0

                # D. WRITE TO IDLE CSV IMMEDIATELY
                idle_writer.writerow([frame_idx, track_id, track_data[track_id]['idle_frames']])
                csv_idle_obj.flush()

                # E. VISUAL FILTER
                if not track.is_confirmed() or track.time_since_update > 1:
                    continue
                
                # Vest processing
                label = track.det_class
                h_box, w_box = y2 - y1, x2 - x1
                # Bounds safety check for cropping
                cy1, cy2 = max(0, y1 + int(h_box*0.2)), min(TARGET_HEIGHT, y1 + int(h_box*0.8))
                cx1, cx2 = max(0, x1 + int(w_box*0.2)), min(TARGET_WIDTH, x1 + int(w_box*0.8))
                vest_crop = frame[cy1:cy2, cx1:cx2]

                if vest_crop.size > 0:
                    if label == 'Safety Vest':
                        current_color = get_vest_color(vest_crop)
                        track_data[track_id]['history'].append(current_color)
                        final_label = max(set(track_data[track_id]['history']), key=track_data[track_id]['history'].count)
                        box_color = (0, 255, 0)
                        frame_vests += 1
                    else:
                        final_label = "!!! NO VEST !!!"
                        box_color = (0, 0, 255)
                        frame_no_vests += 1
                    
                    frame_total += 1
                    fontScale = max(0.4, h_box/400)
                    cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                    cv2.putText(frame, f"ID {track_id}: {final_label}", (x1, y1-10),
                                cv2.FONT_HERSHEY_SIMPLEX, fontScale, box_color, 2)

                if track_data[track_id]['idle_frames'] >= IDLE_FRAME_LIMIT:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 3)
                    cv2.putText(frame, "IDLE", (x1, y1-25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            # ---------- 5. IDLE CLUSTER DETECTION & RECORDING ----------
            idle_clusters = []
            # Find tracks that are currently idle AND confirmed/visible
            active_idle_ids = [t.track_id for t in tracks_out if track_data[t.track_id]['idle_frames'] >= IDLE_FRAME_LIMIT]

            for i in range(len(active_idle_ids)):
                for j in range(i + 1, len(active_idle_ids)):
                    tid1, tid2 = active_idle_ids[i], active_idle_ids[j]
                    c1 = track_data[tid1]['history_centers'][-1]
                    c2 = track_data[tid2]['history_centers'][-1]
                    
                    if distance(c1, c2) < (TARGET_HEIGHT * PROXIMITY_RATIO): 
                        idle_clusters.append((c1, c2))

            cluster_count = len(idle_clusters)

            # Draw Cluster Visuals
            for c1, c2 in idle_clusters:
                cv2.line(frame, c1, c2, (0, 165, 255), 3)
                mid_point = ((c1[0] + c2[0]) // 2, (c1[1] + c2[1]) // 2)
                cv2.putText(frame, "CLUSTER", mid_point, cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)

    # ---------- 6. DASHBOARD DISPLAY ----------
            # Calculate compliance once
            compliance = (frame_vests / frame_total * 100) if frame_total > 0 else 0
            
            # Dashboard background
            cv2.rectangle(frame, (10, 10), (380, 160), (0, 0, 0), -1)
            
            # Dashboard Text
            cv2.putText(frame, f"Total Workers: {frame_total}", (20, 35), 0, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, f"Compliant: {frame_vests}", (20, 60), 0, 0.7, (0, 255, 0), 2)
            cv2.putText(frame, f"Violations: {frame_no_vests}", (20, 85), 0, 0.7, (0, 0, 255), 2)
            cv2.putText(frame, f"Compliance: {compliance:.1f}%", (20, 110), 0, 0.7, (0, 255, 255), 2)
            cv2.putText(frame, f"Idle Clusters: {cluster_count}", (20, 140), 0, 0.7, (0, 165, 255), 2)

            # ALERT Logic
            if compliance < COMPLIANCE_LIMIT and frame_total > 0:
                alert_counter += 1
            else:
                alert_counter = 0

            if alert_counter >= ALERT_THRESHOLD:
                cv2.putText(frame, "!!! LOW PPE COMPLIANCE !!!", (TARGET_WIDTH//2-200, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

            # ---------- 7. FINAL OUTPUT (ONLY ONCE) ----------
            # Write to the main CSV
            writer.writerow([frame_idx, frame_total, frame_vests, frame_no_vests, round(compliance, 2), cluster_count])
            csv_writer_obj.flush()
            
            # Write the frame to the video
            out.write(frame)
            
            # Increment index exactly once
            frame_idx += 1

    finally:
        cap.release()
        out.release()
        csv_writer_obj.close()
        csv_idle_obj.close()
        print(f"Done! Outputs saved in runs/detect/")

In [6]:
#videos = ["Factory_Meet.mp4","Bad_PPE.mp4","Tiny.mp4","Blue.mp4","Mix.mp4","Multi_Concrete.mp4",
#                "Sepia.mp4","X_Pattern.mp4","Mix_Forklift.mp4","Cluster.mp4",
#                 "Office.mp4","Talking.mp4","Sitting","Pose"]
videos = ["Bad_PPE.mp4"]
for vid in videos:
    process_video(vid)

Processing Bad_PPE.mp4 at 1280x720...
Done! Outputs saved in runs/detect/


In [ ]:
#input_video = "Factory_Meet.mp4" # Video of employees with PPE in meeting
#input_video = "Bad_PPE.mp4" # Multiple people not wearing PPE
#input_video = "Tiny.mp4" # Zoomed out timelapse of a construction site
#input_video = "Blue.mp4" # Multiple people wearing blue vests
#input_video = "Mix.mp4" # Mix of people with and without PPE
#input_video = "Multi_Concrete.mp4" # Yellow and orange vests
#input_video = "Sepia.mp4" # Timelapse factory footage in poor lighting
#input_video = "X_Pattern.mp4" # Video of guy in a non-standard vest
#input_video = "Mix_Forklift.mp4" # Multiple people in orange and yellow vests
#input_video = "Cluster.mp4" # Three people not in PPE clustering together
#input_video = "Office.mp4" # Office Employees clustering
#input_video = "Talking.mp4" # Two Employees talking with supervisor
#input_video = "Sitting" # three employees sitting
#input_video = "Pose" #Two employees posing